# Formativa 2 - Modelamiento Predictivo mediante Regresión Logística

**Asignatura:** MCDI501 – Estadística Computacional para la Toma de Decisiones

**Magíster en Ciencia de Datos e Inteligencia Artificial**

**Universidad Andrés Bello**

---

## Objetivo

Desarrollar un modelo de regresión logística para predecir el riesgo de enfermedad coronaria a diez años utilizando el conjunto de datos *Framingham Heart Study*, integrando explícitamente los resultados obtenidos durante la Sumativa 1 y la Sumativa 2.

---

## Integración con las fases anteriores

Durante la **Sumativa 1** se realizó el análisis exploratorio de datos, la estadística descriptiva, la estimación puntual, la construcción de intervalos de confianza y las pruebas de hipótesis.

Durante la **Sumativa 2** se validaron dichos resultados mediante técnicas de remuestreo (bootstrap), pruebas de permutación, simulación Monte Carlo y análisis de robustez.

En esta Formativa 2 se utilizarán estos resultados para fundamentar la selección de variables, el tratamiento de los datos y la construcción del primer modelo predictivo mediante regresión logística, preparando la base para la Sumativa 3.

# 1. Importación de librerías

In [2]:
# ==========================================
# Librerías generales
# ==========================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

# ==========================================
# Scikit-learn
# ==========================================

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    roc_auc_score
)

# ==========================================
# Statsmodels
# ==========================================

import statsmodels.api as sm

from statsmodels.stats.outliers_influence import (
    variance_inflation_factor
)

# ==========================================
# Configuración general
# ==========================================

np.random.seed(42)

plt.rcParams["figure.figsize"] = (8,5)

print("=" * 60)
print("Librerías cargadas correctamente")
print("=" * 60)
print(f"NumPy        : {np.__version__}")
print(f"Pandas       : {pd.__version__}")
print(f"Scikit-learn : {__import__('sklearn').__version__}")
print(f"Statsmodels  : {sm.__version__}")

Librerías cargadas correctamente
NumPy        : 2.4.6
Pandas       : 3.0.3
Scikit-learn : 1.9.0
Statsmodels  : 0.14.6


# 2. Carga del conjunto de datos

## Carga y verificación del conjunto de datos

En esta sección se carga el conjunto de datos **Framingham Heart Study**, utilizado durante todas las fases del proyecto.

Se verifica que el archivo exista correctamente y posteriormente se inspeccionan sus dimensiones, estructura y variables disponibles antes de iniciar el proceso de modelamiento.

In [7]:
# ==========================================
# Carga del conjunto de datos
# ==========================================

df = pd.read_csv(DATA_PATH)

print("=" * 60)
print("Conjunto de datos cargado correctamente")
print("=" * 60)

print(f"Número de filas     : {df.shape[0]}")
print(f"Número de columnas  : {df.shape[1]}")

Conjunto de datos cargado correctamente
Número de filas     : 4238
Número de columnas  : 16


## Inspección inicial

Antes de preparar los datos para el modelamiento se revisa la estructura general del conjunto de datos, incluyendo nombres de variables, tipos de datos y una muestra de los primeros registros.

In [8]:
# ==========================================
# Inspección inicial
# ==========================================

display(df.head())

print("\nTipos de datos\n")
display(df.dtypes)

print("\nResumen del DataFrame\n")
df.info()

,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,1,39,4.0,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0
1,0,46,2.0,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,76.0,0
2,1,48,1.0,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,70.0,0
3,0,61,3.0,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,103.0,1
4,0,46,3.0,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,85.0,0



Tipos de datos



male                 int64
age                  int64
education          float64
currentSmoker        int64
cigsPerDay         float64
BPMeds             float64
prevalentStroke      int64
prevalentHyp         int64
diabetes             int64
totChol            float64
sysBP              float64
diaBP              float64
BMI                float64
heartRate          float64
glucose            float64
TenYearCHD           int64
dtype: object


Resumen del DataFrame

<class 'pandas.DataFrame'>
RangeIndex: 4238 entries, 0 to 4237
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   male             4238 non-null   int64  
 1   age              4238 non-null   int64  
 2   education        4133 non-null   float64
 3   currentSmoker    4238 non-null   int64  
 4   cigsPerDay       4209 non-null   float64
 5   BPMeds           4185 non-null   float64
 6   prevalentStroke  4238 non-null   int64  
 7   prevalentHyp     4238 non-null   int64  
 8   diabetes         4238 non-null   int64  
 9   totChol          4188 non-null   float64
 10  sysBP            4238 non-null   float64
 11  diaBP            4238 non-null   float64
 12  BMI              4219 non-null   float64
 13  heartRate        4237 non-null   float64
 14  glucose          3850 non-null   float64
 15  TenYearCHD       4238 non-null   int64  
dtypes: float64(9), int64(7)
memory usage: 529.9 KB


## Resumen de la calidad de los datos

La evaluación detallada de la calidad del conjunto de datos fue desarrollada durante la Sumativa 1.

Los principales hallazgos fueron:

- Existencia de valores faltantes en algunas variables clínicas.
- Presencia de valores atípicos principalmente en variables continuas.
- Variables numéricas y categóricas correctamente identificadas.
- Conjunto de datos apto para el desarrollo de modelos predictivos.

Durante esta actividad se utilizarán dichos resultados para definir la estrategia de preparación de datos previa al modelamiento.

## Configuración del proyecto

En esta sección se definen las rutas de trabajo utilizadas durante la Formativa 2.

La configuración permite localizar automáticamente el conjunto de datos utilizado en las fases anteriores y crear las carpetas de resultados necesarias para el desarrollo del proyecto.

In [4]:
# ==========================================
# Configuración de rutas del proyecto
# ==========================================

from pathlib import Path

# Buscar automáticamente la carpeta raíz del proyecto
PROJECT_ROOT = Path.cwd()

while PROJECT_ROOT.name != "mcdi501_grupo6":
    PROJECT_ROOT = PROJECT_ROOT.parent

# ==========================================
# Dataset
# ==========================================

DATA_PATH = PROJECT_ROOT / "datos" / "original" / "framingham.csv"

# ==========================================
# Carpetas de la Formativa 2
# ==========================================

RESULTS_DIR = PROJECT_ROOT / "F4" / "formativa2" / "results"

FIGURES_DIR = PROJECT_ROOT / "F4" / "formativa2" / "figures"

# Crear carpetas si no existen
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ==========================================
# Mostrar configuración
# ==========================================

print("=" * 60)
print("Configuración del proyecto")
print("=" * 60)

print(f"Proyecto    : {PROJECT_ROOT}")
print(f"Dataset     : {DATA_PATH}")
print(f"Resultados  : {RESULTS_DIR}")
print(f"Figuras     : {FIGURES_DIR}")

Configuración del proyecto
Proyecto    : C:\users\pablo\documents\mcdi501_grupo6
Dataset     : C:\users\pablo\documents\mcdi501_grupo6\datos\original\framingham.csv
Resultados  : C:\users\pablo\documents\mcdi501_grupo6\F4\formativa2\results
Figuras     : C:\users\pablo\documents\mcdi501_grupo6\F4\formativa2\figures


## Configuración general del análisis

Se definen los parámetros globales que serán utilizados durante todo el proceso de modelamiento. Esto permite mantener la reproducibilidad y consistencia del análisis en todas las etapas.

In [5]:
# ==========================================
# Parámetros generales del análisis
# ==========================================

# Semilla para reproducibilidad
SEED = 42

# Proporción del conjunto de prueba (30 %)
TEST_SIZE = 0.30

# Nivel de significancia
ALPHA = 0.05

print("=" * 60)
print("Parámetros del análisis")
print("=" * 60)
print(f"Semilla aleatoria      : {SEED}")
print(f"Conjunto de prueba     : {TEST_SIZE:.0%}")
print(f"Nivel de significancia : {ALPHA}")

Parámetros del análisis
Semilla aleatoria      : 42
Conjunto de prueba     : 30%
Nivel de significancia : 0.05


# 3. Integración de resultados de la Sumativa 1 y Sumativa 2

## Integración metodológica de las fases anteriores

La construcción del modelo predictivo no constituye un análisis independiente, sino la continuación del trabajo desarrollado durante las fases anteriores del proyecto.

### Resultados incorporados desde la Sumativa 1

Durante la Sumativa 1 se identificaron las principales características estadísticas del conjunto de datos mediante:

- Estadística descriptiva.
- Análisis exploratorio de datos.
- Estimación puntual e intervalos de confianza.
- Pruebas de hipótesis.
- Identificación de valores faltantes.
- Identificación de valores atípicos.
- Análisis de correlaciones entre variables clínicas.

### Resultados incorporados desde la Sumativa 2

Durante la Sumativa 2 se validaron dichos resultados mediante métodos computacionales, incorporando:

- Bootstrap para validar intervalos de confianza.
- Test de permutación para validar pruebas de hipótesis.
- Bootstrap de correlaciones para evaluar estabilidad.
- Simulación Monte Carlo.
- Análisis de sensibilidad frente a valores atípicos.
- Validación de parámetros robustos para el modelamiento.

Todos estos antecedentes servirán como fundamento para la selección de variables predictoras y la construcción del modelo de regresión logística.

## Variables seleccionadas para el modelamiento

La selección inicial de variables predictoras se fundamenta en los resultados obtenidos durante las fases anteriores.

Las variables fueron elegidas considerando:

- Resultados de las pruebas de hipótesis de la Sumativa 1.
- Correlaciones validadas durante la Sumativa 2.
- Robustez observada mediante bootstrap y análisis de sensibilidad.
- Relevancia clínica de cada predictor.

La justificación estadística de cada variable será presentada antes del ajuste del modelo.

# 4. Preparación de datos para el modelamiento

# 5. Construcción del modelo de regresión logística

# 6. Evaluación del desempeño predictivo

# 7. Síntesis y reflexión